# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [13]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [14]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [15]:
print("Shape:", df.shape)
print("-" * 40)

print("Data Types:\n", df.dtypes)
print("-" * 40)

print("Null Count per Column:\n", df.isna().sum())
print("-" * 40)

duplicate_count = df.duplicated().sum()
print("Exact Duplicate Rows:", duplicate_count)

Shape: (8, 6)
----------------------------------------
Data Types:
 order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object
----------------------------------------
Null Count per Column:
 order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64
----------------------------------------
Exact Duplicate Rows: 1


**What is wrong with this data?** List at least five specific problems:

1. _..._
2. _..._
3. _..._
4. _..._
5. _..._

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [16]:
removed = df.duplicated().sum()   # TODO: how many duplicates were there?
clean = df.drop_duplicates().copy()  # TODO: df with duplicates dropped, copied

# TODO: log('duplicates', 'dropped exact duplicate rows', removed)
log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [17]:
clean['price'] = clean['price'].astype(str).str.replace('$', '', regex=False).str.strip().astype(float)

assert clean['price'].dtype == float

log('price', 'converted price text to float after stripping $ and whitespace', len(clean))

[price] converted price text to float after stripping $ and whitespace (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [18]:
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')
missing = clean['qty'].isna().sum()    # TODO: count of NaN quantities
negative = (clean['qty'] < 0).sum()   # TODO: count of negative quantities

# TODO: apply your decision, then log both separately
clean = clean.dropna(subset=['qty']).copy()
log('qty_missing', 'dropped rows with missing quantity', missing)

log('qty_negative', 'retained negative quantity as return/refund', negative)

[qty_missing] dropped rows with missing quantity (1 row(s))
[qty_negative] retained negative quantity as return/refund (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [19]:
print('before:', sorted(clean['category'].unique()))
before_count = len(clean['category'].unique())

clean['category'] = (
    clean['category']
    .str.lower()
    .str.strip()
    .str.replace('-', '', regex=False)
)

CATEGORY_MAP = {
    'raingear': 'Rain Gear'
}
clean['category'] = clean['category'].replace(CATEGORY_MAP).str.title()

print('after: ', sorted(clean['category'].unique()))
after_count = len(clean['category'].unique())

log('category', f'collapsed categories from {before_count} to {after_count}', len(clean))

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after:  ['Apparel', 'Food', 'Merch', 'Rain Gear']
[category] collapsed categories from 6 to 4 (6 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [20]:
print('before:', sorted(clean['item'].dropna().unique()))
before_count = len(clean['item'].dropna().unique())

clean['item'] = clean['item'].astype(str).str.strip()

ITEM_MAP = {
    'cheese burger': 'Cheeseburger',
    'cheeseburger': 'Cheeseburger',
    'rain poncho': 'Rain Poncho',
    'nan': np.nan,
    'NaN': np.nan
}
clean['item'] = clean['item'].replace(ITEM_MAP)

missing_items = clean['item'].isna().sum()
clean['item'] = clean['item'].fillna('Unknown')

print('after: ', sorted(clean['item'].unique()))
after_count = len(clean['item'].unique())

log('item', f'standardized names and filled {missing_items} missing item(s) with Unknown', len(clean))

before: ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
after:  ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt', 'Unknown']
[item] standardized names and filled 1 missing item(s) with Unknown (6 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [21]:
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce')

missing_ts = clean['ts'].isna().sum()

clean['hour'] = clean['ts'].dt.hour

log('timestamps', 'parsed ts column to datetime and extracted hour', missing_ts)

[timestamps] parsed ts column to datetime and extracted hour (3 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [22]:
assert clean['price'].dtype == float, "Price is not a float"
assert clean['qty'].dtype == float, "Qty is not numeric"
assert pd.api.types.is_datetime64_any_dtype(clean['ts']), "ts is not datetime"
assert clean['price'].isna().sum() == 0, "Null values found in price"
assert clean['category'].str.isupper().sum() == 0, "Unnormalized categories present"
assert clean.duplicated().sum() == 0, "Duplicate rows exist"

clean['revenue'] = clean['qty'] * clean['price']

print(f"Total Rows: {len(clean)}")
print(f"Total Units Sold: {clean['qty'].sum()}")
print(f"Total Revenue: ${clean['revenue'].sum():.2f}")
print(f"Distinct Categories: {clean['category'].nunique()}")

Total Rows: 6
Total Units Sold: 7.0
Total Revenue: $88.50
Distinct Categories: 4


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [23]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price,converted price text to float after stripping ...,7
2,qty_missing,dropped rows with missing quantity,1
3,qty_negative,retained negative quantity as return/refund,1
4,category,collapsed categories from 6 to 4,6
5,item,standardized names and filled 1 missing item(s...,6
6,timestamps,parsed ts column to datetime and extracted hour,3


**The decision that mattered most:** _..._

**Revenue with it:** _..._  **Revenue without it:** _..._

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [24]:
# Checkpoint
rows_after = len(clean)            # TODO
revenue_after = clean['revenue'].sum()         # TODO
biggest_decision = 'kept negative quantity as refund instead of dropping it'    # TODO: which choice moved the number most
revenue_other_way = clean[clean['qty'] > 0]['revenue'].sum()     # TODO: the total if you had chosen differently

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 6
revenue: 88.5
decision that mattered: kept negative quantity as refund instead of dropping it
revenue the other way: 106.5
